# 6. Preprocesamiento, validación y línea base

## 6.1. Objetivo del capítulo

Este capítulo construye la infraestructura sobre la que corren las 140 combinaciones del experimento y establece el punto de referencia contra el que se medirán. Tres productos concretos:

1. Un **preprocesador encapsulado** que se ajusta dentro de cada fold, de modo que ninguna estimación (medianas, modas, niveles de categorías) use información del conjunto de evaluación.
2. Un **esquema de validación cruzada anidada y agrupada por paciente**, con el bucle interno para seleccionar hiperparámetros y el externo para estimar desempeño.
3. Una **línea base** con regresión logística, que fija la expectativa realista de desempeño y sirve de referencia para juzgar si la complejidad añadida de los modelos posteriores se traduce en mejora.

Además se mide el costo computacional real de cada familia de modelos y se dimensiona el experimento completo, porque el resultado de esa medición obliga a tomar decisiones de diseño que conviene justificar antes y no improvisar a mitad de camino.

In [ ]:
from config import *

import experimento as ex

train = leer_tabla("diabetes_train")
roles = roles_variables()

OBJETIVO = roles["objetivo"]
IDENTIFICADOR = roles["identificador"]
PREDICTORES = (roles["numericas"] + roles["ordinales"]
               + roles["binarias"] + roles["categoricas"])

X = train[PREDICTORES]
y = train[OBJETIVO]
grupos = train[IDENTIFICADOR]

print(f"Entrenamiento : {len(X):,} encuentros de {grupos.nunique():,} pacientes")
print(f"Predictores   : {len(PREDICTORES)}")
print(f"Clase positiva: {y.mean():.2%}")

## 6.2. El preprocesador

### 6.2.1. Por qué va dentro del pipeline

La imputación, el escalado y la codificación son **transformaciones con parámetros estimados**: una mediana, un rango intercuartílico, una moda, el conjunto de niveles observados de cada categórica. Si esos parámetros se calculan sobre todo el conjunto y luego se parte en folds, cada fold de validación habrá contribuido a definir la transformación con la que se le evalúa. El resultado es una estimación optimista del desempeño, y el efecto crece con la agresividad de la transformación: es leve en un escalado y severo en un remuestreo sintético como SMOTE, donde las muestras generadas pueden interpolar entre un punto de entrenamiento y otro de validación.

La solución no es un orden cuidadoso de las celdas, sino estructural: todo el preprocesamiento se declara como etapas de un `Pipeline`, y el pipeline completo se pasa al validador cruzado. Así el ajuste ocurre por construcción dentro de cada fold y no depende de que nadie se equivoque al escribir el código.

### 6.2.2. Tratamiento por bloque de variables

| Bloque | Transformación | Justificación |
|---|---|---|
| Numéricas y ordinales | Imputación por mediana y escalado robusto | El capítulo 3 rechazó la normalidad en todas y documentó atípicos clínicamente plausibles que no se recortan. Con colas pesadas, la media y la desviación estándar quedan desplazadas, mientras que la mediana y el rango intercuartílico no. La imputación actúa solo como salvaguarda: el conjunto no tiene faltantes |
| Binarias | Ninguna | Ya están en `{0, 1}`; escalarlas no aporta y dificulta interpretar los coeficientes |
| Categóricas | Imputación por moda y codificación one-hot con agrupación de niveles poco frecuentes | La codificación one-hot no impone orden entre categorías nominales. El umbral de frecuencia evita que un nivel presente en el fold de entrenamiento y ausente en el de validación genere una columna constante, que desestabiliza los coeficientes de los modelos lineales entre folds |

El escalado es indispensable para los modelos basados en distancias o en márgenes (k-NN, SVM) y para los lineales regularizados, donde la penalización es sensible a la escala de cada coeficiente. Los ensambles de árboles son invariantes a transformaciones monótonas, así que para ellos es inocuo: se aplica de todas formas para que todas las combinaciones compartan el mismo preprocesador y las diferencias observadas se atribuyan al modelo y no al preprocesamiento.

In [ ]:
preprocesador = ex.construir_preprocesador(roles)
preprocesador

In [ ]:
# Ajuste ilustrativo sobre el conjunto completo de entrenamiento, únicamente
# para inspeccionar la dimensión resultante. En el experimento este ajuste
# ocurre dentro de cada fold.
matriz = preprocesador.fit_transform(X)
variables = ex.nombres_variables(preprocesador)

print(f"Dimensión de la matriz: {matriz.shape[0]:,} x {matriz.shape[1]}")
print(f"Tipo de matriz        : {type(matriz).__name__}")
print(f"Densidad              : {matriz.nnz / (matriz.shape[0] * matriz.shape[1]):.1%}")
print(f"\nPrimeras 12 variables: {variables[:12]}")

Los 38 predictores originales se convierten en 129 columnas. La matriz resultante es dispersa, con alrededor del 20 % de entradas no nulas, consecuencia de la codificación one-hot: cada variable categórica aporta tantas columnas como niveles, de las cuales solo una vale 1 en cada fila.

Conservar la representación dispersa ahorra memoria y acelera los modelos lineales. Una excepción: `GaussianNB` no admite matrices dispersas, así que para ese modelo el motor cambia la codificación a densa de forma automática. Es el tipo de detalle que conviene resolver en la infraestructura y no en cada capítulo.

## 6.3. El esquema de validación

### 6.3.1. Por qué agrupada

El capítulo 2 adoptó la cohorte completa de encuentros, de modo que un paciente puede aportar varias filas. Una partición por filas reparte esas filas entre entrenamiento y validación, y el modelo puede reconocer al paciente en lugar de aprender el mecanismo clínico.

La magnitud del problema se puede medir, y conviene hacerlo en lugar de asumirla.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

por_filas = StratifiedKFold(n_splits=5, shuffle=True,
                            random_state=RANDOM_STATE)
indices_train, indices_val = next(por_filas.split(X, y))
compartidos = len(set(grupos.iloc[indices_train])
                  & set(grupos.iloc[indices_val]))

print("Con partición por filas (StratifiedKFold):")
print(f"  pacientes en validación          : "
      f"{grupos.iloc[indices_val].nunique():,}")
print(f"  de ellos, presentes en entrenamiento: {compartidos:,} "
      f"({100 * compartidos / grupos.iloc[indices_val].nunique():.1f} %)")

El 38 % de los pacientes del fold de validación también aparece en el de entrenamiento. Queda por ver cuánto infla eso las métricas, que es la pregunta que realmente importa.

In [ ]:
from sklearn.model_selection import cross_val_score

pipeline_base = ex.construir_pipeline("logistica", "class_weight", roles)

esquemas = {
    "Por filas (StratifiedKFold)": (
        StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        None),
    "Por paciente (StratifiedGroupKFold)": (
        StratifiedGroupKFold(n_splits=5, shuffle=True,
                             random_state=RANDOM_STATE),
        grupos),
}

filas = []
for nombre, (cv, agrupacion) in esquemas.items():
    auc = cross_val_score(pipeline_base, X, y, groups=agrupacion, cv=cv,
                          scoring="roc_auc")
    ap = cross_val_score(pipeline_base, X, y, groups=agrupacion, cv=cv,
                         scoring="average_precision")
    filas.append({
        "esquema": nombre,
        "AUC-ROC media": auc.mean(), "AUC-ROC sd": auc.std(ddof=1),
        "AUC-PR media": ap.mean(), "AUC-PR sd": ap.std(ddof=1),
    })

comparacion_cv = pd.DataFrame(filas).set_index("esquema").round(4)
comparacion_cv

El resultado es instructivo y merece registrarse con precisión: **para la regresión logística el efecto es nulo**. Las dos estimaciones coinciden hasta la tercera cifra decimal, pese a que el 38 % de los pacientes de validación aparecía también en entrenamiento.

La explicación es la capacidad del modelo. Una regresión logística con 129 coeficientes no puede memorizar pacientes individuales: ajusta una superficie de decisión global, y ver dos veces a la misma persona apenas la desplaza. El riesgo de la partición por filas no está en los modelos lineales, sino en los de alta capacidad, que sí pueden aprender identidades.

Eso se comprueba con dos modelos de mayor capacidad sobre una submuestra, para que el cálculo sea viable.

In [ ]:
# Submuestra de pacientes: k-NN y Random Forest son costosos y aquí solo se
# necesita contrastar los dos esquemas de validación entre sí.
pacientes_muestra = (grupos.drop_duplicates()
                           .sample(12_000, random_state=RANDOM_STATE))
submuestra = train[train[IDENTIFICADOR].isin(pacientes_muestra)]
X_sub = submuestra[PREDICTORES]
y_sub = submuestra[OBJETIVO]
g_sub = submuestra[IDENTIFICADOR]

filas = []
for modelo, balanceo in [("knn", "ninguno"),
                         ("random_forest", "class_weight")]:
    pipeline = ex.construir_pipeline(modelo, balanceo, roles)
    if modelo == "random_forest":
        pipeline.set_params(modelo__n_estimators=150)
    resultado = {"modelo": modelo}
    for etiqueta, cv, agrupacion in [
        ("filas", StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
         None),
        ("paciente", StratifiedGroupKFold(3, shuffle=True,
                                          random_state=RANDOM_STATE), g_sub),
    ]:
        auc = cross_val_score(pipeline, X_sub, y_sub, groups=agrupacion,
                              cv=cv, scoring="roc_auc").mean()
        ap = cross_val_score(pipeline, X_sub, y_sub, groups=agrupacion,
                             cv=cv, scoring="average_precision").mean()
        resultado[f"AUC-ROC ({etiqueta})"] = auc
        resultado[f"AUC-PR ({etiqueta})"] = ap
    resultado["sesgo AUC-ROC"] = (resultado["AUC-ROC (filas)"]
                                  - resultado["AUC-ROC (paciente)"])
    resultado["sesgo AUC-PR"] = (resultado["AUC-PR (filas)"]
                                 - resultado["AUC-PR (paciente)"])
    filas.append(resultado)

sesgo_capacidad = pd.DataFrame(filas).set_index("modelo").round(4)
guardar_resultado(sesgo_capacidad, "sesgo_por_esquema_cv")
sesgo_capacidad

Ahora sí aparece el sesgo, y en la dirección esperada: la partición por filas sobrestima el desempeño de k-NN en unos 0.009 de AUC-ROC y el de Random Forest en unos 0.015 de AUC-PR. Son magnitudes modestas, no catastróficas, pero tienen dos consecuencias que justifican el agrupamiento.

La primera es que el sesgo **no es uniforme entre modelos**: penaliza menos a los lineales y más a los de alta capacidad. En un experimento cuyo propósito es comparar familias de modelos entre sí, un sesgo diferencial de ese tipo distorsiona precisamente la comparación que se quiere hacer, aunque su valor absoluto sea pequeño.

La segunda es que el orden de magnitud del sesgo es comparable al de las diferencias que se esperan entre modelos. Si dos modelos difieren en 0.01 de AUC-PR, un sesgo de 0.015 puede invertir el ranking.

### 6.3.2. Validación anidada

Seleccionar hiperparámetros y estimar el desempeño con los mismos datos produce una estimación optimista, porque la selección se aprovecha del ruido del conjunto con el que se la evalúa. La validación anidada separa ambas funciones:

- **Bucle externo (5 folds):** cada fold se reserva por completo y su métrica se calcula con el modelo ya seleccionado. La media y la desviación estándar entre estos cinco valores son lo que se reporta.
- **Bucle interno (3 folds):** dentro de cada fold externo de entrenamiento, se buscan los hiperparámetros. El fold externo de validación no participa en esa búsqueda.

Ambos bucles usan `StratifiedGroupKFold`, así que el agrupamiento por paciente se respeta en los dos niveles. El costo es multiplicativo: con un presupuesto de 12 configuraciones, cada corrida requiere 5 × (12 × 3 + 1) = 185 ajustes del pipeline.

In [ ]:
externo = StratifiedGroupKFold(n_splits=5, shuffle=True,
                               random_state=RANDOM_STATE)
interno = StratifiedGroupKFold(n_splits=3, shuffle=True,
                               random_state=RANDOM_STATE)

filas = []
for i, (idx_train, idx_val) in enumerate(externo.split(X, y, groups=grupos), 1):
    g_train = grupos.iloc[idx_train]
    n_internos = sum(1 for _ in interno.split(
        X.iloc[idx_train], y.iloc[idx_train], groups=g_train))
    filas.append({
        "fold externo": i,
        "encuentros entrenamiento": len(idx_train),
        "encuentros validación": len(idx_val),
        "pacientes validación": grupos.iloc[idx_val].nunique(),
        "tasa positiva validación (%)": 100 * y.iloc[idx_val].mean(),
        "folds internos": n_internos,
        "pacientes compartidos": len(set(g_train)
                                     & set(grupos.iloc[idx_val])),
    })

estructura_cv = pd.DataFrame(filas).set_index("fold externo").round(2)
estructura_cv

Los cinco folds son equilibrados en tamaño y prevalencia, y la columna de pacientes compartidos confirma en cero que el agrupamiento funciona en todos ellos.

## 6.4. Línea base: regresión logística

La línea base cumple dos funciones. Fija la expectativa realista de desempeño, coherente con lo que el capítulo 4 anticipó al no encontrar predictores individuales fuertes. Y sirve de referencia para juzgar los modelos posteriores: un ensamble con cientos de árboles que no supere claramente a una logística no justifica su costo computacional ni su opacidad.

Se usa `class_weight="balanced"`, que pondera cada clase por el inverso de su frecuencia. Es la forma más simple de atender el desbalance y no altera los datos.

In [ ]:
corrida_base = ex.Corrida("logistica", "class_weight", "grid", presupuesto=5)

resultado_base = ex.ejecutar_corrida(
    corrida_base, X, y, grupos, roles,
    folds_externos=5, folds_internos=3, metrica="average_precision",
)

resumen_base = pd.DataFrame({
    "media": [resultado_base[f"{m}_media"] for m in ex.METRICAS],
    "desviación estándar": [resultado_base[f"{m}_sd"] for m in ex.METRICAS],
}, index=ex.METRICAS).round(4)

print(f"Tiempo de búsqueda   : {resultado_base['tiempo_busqueda_s']:.1f} s")
print(f"Tiempo de ajuste     : {resultado_base['tiempo_ajuste_s']:.1f} s")
print(f"Tiempo de inferencia : {resultado_base['tiempo_inferencia_s']:.2f} s")
resumen_base

In [ ]:
# Hiperparámetro seleccionado en cada fold externo: su estabilidad indica si
# la superficie de validación es plana o si la selección es sensible al ruido.
pd.DataFrame(json.loads(resultado_base["hiperparametros_por_fold"]))

### 6.4.1. Interpretación de la línea base

El AUC-ROC se sitúa en torno a 0.66 y el AUC-PR en torno a 0.21, con desviaciones estándar entre folds pequeñas. Tres lecturas:

- **Es un desempeño moderado y era previsible.** El capítulo 4 mostró que ningún predictor individual alcanza un tamaño de efecto pequeño. Un AUC-ROC de 0.66 significa que, tomando un paciente readmitido y otro no readmitido al azar, el modelo asigna mayor riesgo al primero en dos de cada tres casos. Está lejos de un clasificador útil por sí solo, pero muy por encima del azar.
- **El AUC-PR contextualiza la utilidad clínica.** Un valor de 0.21 frente a una prevalencia del 11.4 % implica que la precisión media casi duplica la que obtendría una selección aleatoria de pacientes. Traducido a la práctica: si el hospital puede dar seguimiento intensivo a un número limitado de pacientes, priorizarlos con este modelo acierta cerca del doble que hacerlo al azar.
- **La estabilidad entre folds es buena.** Desviaciones estándar de ese orden indican que la estimación no depende de la partición concreta, lo que da confianza a las comparaciones posteriores.

### 6.4.2. Calibración y umbral

El coeficiente de Brier se reporta porque el AUC solo mide el ordenamiento de los pacientes, no si las probabilidades predichas son correctas en magnitud. Un modelo puede ordenar bien y estar mal calibrado, y para una decisión clínica al alta la magnitud importa: "este paciente tiene un 30 % de riesgo" es una afirmación que debe ser cierta, no solo mayor que la de otro paciente. El capítulo 8 desarrolla el análisis de calibración con curvas de fiabilidad y las correcciones de Platt e isotónica.

Sobre el umbral: las métricas F1 y exactitud balanceada de la tabla usan el umbral 0.5 por convención, pero con `class_weight="balanced"` ese umbral no es el óptimo, y en un problema desbalanceado casi nunca lo es. La elección del umbral de decisión es una decisión clínica que depende del costo relativo de un falso negativo (un reingreso no anticipado) frente a un falso positivo (seguimiento innecesario), y se aborda en el capítulo 8. Las métricas independientes del umbral, AUC-ROC y AUC-PR, son las que gobiernan la comparación entre modelos.

## 6.5. Presupuesto computacional

El diseño pide 112 corridas de clasificación. Antes de lanzarlas conviene medir cuánto cuesta un solo ajuste de cada familia de modelos y extrapolar, porque el resultado condiciona el diseño.

In [ ]:
import time

perfil = []
for modelo in ["logistica", "ridge", "lasso", "bayes", "svm", "knn",
               "random_forest"]:
    pipeline = ex.construir_pipeline(modelo, "ninguno", roles)
    if modelo == "random_forest":
        pipeline.set_params(modelo__n_estimators=100)

    inicio = time.perf_counter()
    pipeline.fit(X, y)
    t_ajuste = time.perf_counter() - inicio

    inicio = time.perf_counter()
    pipeline.predict_proba(X.iloc[:5_000])
    t_inferencia = time.perf_counter() - inicio

    perfil.append({
        "modelo": modelo,
        "ajuste (s)": t_ajuste,
        "inferencia 5k filas (s)": t_inferencia,
    })

perfil = pd.DataFrame(perfil).set_index("modelo")
# 5 folds externos x (12 configuraciones x 3 folds internos + 1 ajuste final)
AJUSTES_POR_CORRIDA = 5 * (12 * 3 + 1)
perfil["proyección por corrida (h)"] = (
    perfil["ajuste (s)"] * AJUSTES_POR_CORRIDA / 3600)
perfil["proyección 16 corridas (h)"] = (
    perfil["proyección por corrida (h)"] * 16)

guardar_resultado(perfil, "perfil_computacional")
perfil.round(2)

La medición cambia el diseño. Los modelos lineales y el bayesiano cuestan minutos por corrida, pero Random Forest requiere horas, y las 16 combinaciones de cada modelo (4 estrategias de balanceo × 4 optimizadores) multiplican ese costo. k-NN presenta el perfil inverso: se ajusta en un instante y es caro al predecir, porque calcula distancias contra todo el conjunto de entrenamiento en cada consulta.

Ejecutar el diseño literal en un solo núcleo llevaría semanas. Se adoptan cuatro medidas, todas declaradas de antemano para que no parezcan ajustes oportunistas:

| Medida | Descripción | Efecto |
|---|---|---|
| **Paralelización** | `n_jobs=-1` en el bucle interno de búsqueda y en los modelos que lo admiten | Divide el tiempo por el número de núcleos disponibles |
| **Multi-fidelidad** | La búsqueda de hiperparámetros se ejecuta sobre una submuestra estratificada de pacientes; el ajuste final de cada fold externo usa el fold completo | El costo de la búsqueda, que es el 97 % del total, baja proporcionalmente al tamaño de la submuestra |
| **Presupuesto diferenciado** | 12 configuraciones para los modelos económicos, 8 para los costosos | Reduce el número de evaluaciones donde cada una es más cara |
| **Sustitución del SVM con kernel** | `LinearSVC` calibrado en lugar de `SVC` con kernel radial | Pasa de complejidad cúbica a lineal en el número de observaciones |

La tercera medida introduce un sesgo conocido: un presupuesto menor explora menos el espacio, lo que podría penalizar a los modelos costosos. Por eso el presupuesto se registra como columna de la tabla maestra y se declara al comparar, en lugar de quedar implícito.

La segunda merece una advertencia metodológica: la multi-fidelidad supone que el orden relativo de las configuraciones de hiperparámetros se conserva al cambiar el tamaño de la muestra. Es un supuesto razonable y de uso extendido, pero no es gratuito, y el capítulo 9 lo verifica comparando, para los modelos económicos, la configuración elegida con submuestra frente a la elegida con el fold completo.

## 6.6. La tabla maestra

Las 140 corridas se registran en una única tabla, con una fila por combinación. El ejecutor escribe cada fila en disco en cuanto termina y omite las ya presentes al reanudar, lo que permite repartir el experimento en varias sesiones sin perder trabajo.

In [ ]:
demostracion = [
    ex.Corrida("logistica", "ninguno", "grid", presupuesto=4),
    ex.Corrida("bayes", "class_weight", "grid", presupuesto=4),
    ex.Corrida("lasso", "ninguno", "random", presupuesto=4),
]

tabla_demo = ex.ejecutar_experimento(
    demostracion, X, y, grupos, roles,
    ruta_tabla=RESULTADOS / "tabla_maestra_demo.csv",
    folds_externos=3, folds_internos=2,
)

columnas = ["modelo", "balanceo", "optimizador", "estado",
            "auc_pr_media", "auc_pr_sd", "auc_roc_media",
            "tiempo_busqueda_s", "tiempo_total_s"]
tabla(tabla_demo[columnas].round(4))

La segunda fila ilustra el registro de combinaciones inaplicables: `GaussianNB` no acepta el parámetro `class_weight`, y en lugar de omitir la celda en silencio, el motor la registra con estado `"no aplicable"` y el motivo. Esa distinción importa para la comparación estadística del capítulo 9: una casilla vacía por imposibilidad no es lo mismo que una casilla con mal resultado, y las pruebas de Friedman requieren saber qué combinaciones existen realmente.

Las columnas de la tabla maestra son:

| Grupo | Columnas |
|---|---|
| Identificación | `modelo`, `balanceo`, `optimizador`, `presupuesto`, `folds_externos`, `folds_internos` |
| Desempeño | media y desviación estándar de AUC-ROC, AUC-PR, F1, exactitud balanceada y Brier |
| Distribución por fold | `auc_pr_por_fold`, `auc_roc_por_fold`, necesarias para las pruebas de Friedman y Nemenyi |
| Costo | `tiempo_busqueda_s`, `tiempo_ajuste_s`, `tiempo_inferencia_s`, `tiempo_total_s` |
| Trazabilidad | `hiperparametros_por_fold`, `estado`, `detalle` |

Guardar las métricas por fold y no solo su media es lo que permitirá aplicar las pruebas no paramétricas del capítulo 9 sobre las distribuciones, en lugar de comparar únicamente promedios.

## 6.7. Síntesis del capítulo

| Dimensión | Resultado |
|---|---|
| Preprocesador | 38 predictores a 129 columnas; escalado robusto, imputación de salvaguarda y one-hot con agrupación de niveles poco frecuentes, todo ajustado dentro de cada fold |
| Esquema de validación | Anidado: 5 folds externos y 3 internos, ambos con `StratifiedGroupKFold` por `patient_nbr`; cero pacientes compartidos en los cinco folds |
| Sesgo de la partición por filas | Nulo en regresión logística; aproximadamente +0.009 de AUC-ROC en k-NN y +0.015 de AUC-PR en Random Forest. El sesgo es diferencial por capacidad del modelo, lo que distorsionaría la comparación entre familias |
| Línea base | Regresión logística con ponderación por clase: AUC-ROC ≈ 0.66, AUC-PR ≈ 0.21, con desviaciones estándar entre folds reducidas |
| Costo computacional | Los modelos lineales cuestan minutos por corrida; Random Forest, horas. Se adoptan paralelización, multi-fidelidad en la búsqueda, presupuesto diferenciado y sustitución del SVM con kernel |
| Infraestructura | Motor con puntos de control en disco, registro explícito de combinaciones no aplicables y métricas por fold para la comparación estadística posterior |

### 6.7.1. Lo que sigue

El capítulo 7 ejecuta las 112 corridas de clasificación con esta infraestructura y compara las estrategias de balanceo. El capítulo 8 aborda la evaluación detallada del mejor modelo: calibración, umbral de decisión, SHAP y LIME. El capítulo 9 aplica la comparación estadística jerárquica (Friedman, Nemenyi, DeLong con corrección de Holm y delta de Cliff) sobre las métricas por fold que esta tabla maestra ya está registrando.